# Lab: Vitis HLS Pragmas and Hardware Structures

---

## 1. Goals

In this lab you’ll use three FIFO implementations inside a Conway’s Game of Life accelerator:

* `fifo_shiftreg_1`: shift register with **compile-time** (static) width.
* `fifo_shiftreg_2`: shift register with **run-time** (programmable) width.
* `fifo_bram`: circular buffer implemented with RAM.

You will:

* Compare performance and hardware resources for the three FIFOs.
* Use HLS pragmas to “fix” the slow `fifo_shiftreg_2` until it behaves similarly to `fifo_shiftreg_1`.
* Observe how pragma usage changes the synthesized hardware (shift registers vs BRAM, parallel vs sequential).

Everything below assumes you already have the provided code in your HLS project.

---

## 2. General Procedure for Each Experiment

For each configuration:

1. Modify the top function `gameoflife_compute` to instantiate the desired FIFO type for `line_1`, `line_2`, and `line_3`.
2. Modify the FIFO classes only as instructed in that experiment.
3. Run C synthesis in Vitis HLS.
4. Record:

   * Latency of `gameoflife_compute`.
   * Initiation interval (II) of the inner `x` loop (if pipelined).
   * LUT, FF, BRAM usage.
   * Estimated maximum frequency.

Keep a table of results; it will be your best friend in the report.

---

## 3. Experiment 1 – Static Shift Register FIFO (fifo_shiftreg_1)

### 3.1 Objective

Use `fifo_shiftreg_1` as the line buffer implementation. This version has a **compile-time fixed width** and typically synthesizes to a very efficient shift register.

### 3.2 Actions

1. In `gameoflife_compute`, ensure that the three line buffers are instantiated using `fifo_shiftreg_1` only (no `fifo_shiftreg_2` or `fifo_bram` in this configuration).
2. Do not add any pragmas to `fifo_shiftreg_1` for this experiment.
3. Leave the rest of the code (loops, `mat3`, etc.) as provided.

### 3.3 Synthesis and Measurement

* Run synthesis.
* Record latency, II, LUT/FF/BRAM usage.
* This is your “good performance, static width” baseline.

---

## 4. Experiment 2 – Programmable Width FIFO (fifo_shiftreg_2) Without Pragmas

### 4.1 Objective

Switch to `fifo_shiftreg_2`, which allows **run-time programmable width**, but with worse default characteristics. You will see how bad it is before touching pragmas.

### 4.2 Actions

1. In `gameoflife_compute`, change the three line buffers so they use `fifo_shiftreg_2` instead of `fifo_shiftreg_1`.

   * Use the runtime `grid_width - 1` as the width argument.
   * Ensure `fifo_shiftreg_1` is not used in this configuration.
2. Inside the `fifo_shiftreg_2` class:

   * Do not add any pragmas yet.
   * Keep the loop in `shift` that iterates from `max_width - 1` down to `1` as is.

### 4.3 Synthesis and Measurement

* Run synthesis.
* Record latency, II, resource usage.
* Compare with Experiment 1:

  * How much slower is it?
  * Does II change?

You should see that the programmable-width version without pragmas is significantly worse.

---

## 5. Experiment 3 – Fixing fifo_shiftreg_2 with Pragmas

Now you will progressively add pragmas to `fifo_shiftreg_2` until its behavior is close to `fifo_shiftreg_1`.

We will turn the sequential, long shift loop into a highly parallel structure.

### 5.1 Step 3.1 – Bind fifo_shiftreg_2 Storage to LUTRAM

**Objective:** Force the `data` array in `fifo_shiftreg_2` to use LUTRAM instead of BRAM, without changing algorithm complexity.

**Location:**

* Inside the `fifo_shiftreg_2` class, in the `private` section, immediately after the line that declares the `data` array (`T data[max_width];`).

**Action:**

Add this line directly after the `data` array declaration in `fifo_shiftreg_2`:

* `#pragma HLS BIND_STORAGE variable=data type=ram_1p impl=lutram`

**Synthesis:**

* Run synthesis.
* Record latency, II, resource usage.
* Compare with Experiment 2:

  * You should see BRAM usage go down, LUT/FF usage change, but latency still large.

---

### 5.2 Step 3.2 – Inline fifo_shiftreg_2::shift

**Objective:** Inline the `shift` function so the tool can optimize it together with the calling code.

**Location:**

* Inside the `fifo_shiftreg_2` class, at the beginning of the `shift` method body, right after the opening brace `{`.

**Action:**

Add this line as the **first statement** inside the `shift` method in `fifo_shiftreg_2`:

* `#pragma HLS INLINE`

**Synthesis:**

* Run synthesis.
* Record latency, II, resource usage.
* The behavior will still be O(max_width), but the logic is now visible to the top function for further optimization.

---

### 5.3 Step 3.3 – Unroll the Shift Loop in fifo_shiftreg_2

**Objective:** Turn the long sequential shift into a parallel shift across all elements.

**Location:**

* Inside the `shift` method of `fifo_shiftreg_2`, in the `for` loop that shifts `data` elements (the loop labeled `fifo_ff_loop`).

**Action:**

Inside that `for` loop, as the first line inside the loop body (immediately after the `{` of the loop), add:

* `#pragma HLS UNROLL`

So the order inside the loop body should be:

1. `#pragma HLS UNROLL`
2. The assignment statements that shift `data[i]`.

Keep:

* The `#pragma HLS INLINE` line at the start of `shift`.
* The `#pragma HLS BIND_STORAGE variable=data type=ram_1p impl=lutram` line after the `data` array declaration.

**Synthesis:**

* Run synthesis.
* Record latency, II, and resource usage.
* Compare with Step 3.2:

  * Latency should drop sharply.
  * LUT/FF usage should increase.
  * BRAM usage remains low.

Now `fifo_shiftreg_2` is acting like a big parallel shift register.

---

### 5.4 Step 3.4 – Partition fifo_shiftreg_2 Data Array

**Objective:** Expose maximum parallelism by fully partitioning the `data` array.

**Location:**

* In the `private` section of `fifo_shiftreg_2`, right after the line where you bound `data` to LUTRAM.

**Action:**

Immediately after the line
`#pragma HLS BIND_STORAGE variable=data type=ram_1p impl=lutram`
add the following pragma:

* `#pragma HLS ARRAY_PARTITION variable=data complete dim=1`

So, in order, you should have:

1. `T data[max_width];`
2. `#pragma HLS BIND_STORAGE variable=data type=ram_1p impl=lutram`
3. `#pragma HLS ARRAY_PARTITION variable=data complete dim=1`

Keep:

* `#pragma HLS INLINE` at the start of `shift`.
* `#pragma HLS UNROLL` inside the shift loop.

**Synthesis:**

* Run synthesis.
* Record latency, II, and resource usage.
* Compare with Step 3.3:

  * Check how LUT/FF usage changes.
  * See whether timing (max frequency) improves or worsens.

At this point, `fifo_shiftreg_2` should be functionally very close in performance to `fifo_shiftreg_1`, but with different area and timing trade-offs.

---

## 6. Experiment 4 – Pipelining Game of Life and Optimizing mat3

Now you optimize the *whole accelerator* around your shiny new FIFO.

### 6.1 Step 4.1 – Pipeline the Inner x Loop

**Objective:** Try to process one cell per clock cycle.

**Location:**

* In `gameoflife_compute`, just before the `for` loop that iterates over `x` (the inner loop labeled something like `gameoflife_x_loop`).

**Action:**

Directly above the `for (int x = ... )` line for the inner loop, add:

* `#pragma HLS PIPELINE II=1`

Keep all the `fifo_shiftreg_2` pragmas from Experiment 3.

**Synthesis:**

* Run synthesis.
* Record the achieved II for the inner `x` loop, latency, and resource usage.
* If II > 1, something in the body is limiting the pipeline (often the neighbor sum).

---

### 6.2 Step 4.2 – Partition mat3

**Objective:** Allow parallel access to all 9 elements of the 3×3 neighborhood.

**Location:**

* In `gameoflife_compute`, at the declaration of the `mat3` array.

**Action:**

Immediately after the line that declares the `mat3` array (the line with `int mat3[9];`), add:

* `#pragma HLS ARRAY_PARTITION variable=mat3 complete dim=1`

This tells HLS that each element of `mat3` can be stored separately and accessed in parallel.

---

### 6.3 Step 4.3 – Unroll the Neighbor Sum Loop

**Objective:** Turn the neighbor sum into a purely combinational adder tree.

**Location:**

* In `gameoflife_compute`, inside the loop that computes the sum of the neighbors, typically the one labeled `compute_sum_loop` that iterates `i` from 0 to 8 and skips index 4.

**Action:**

Inside this neighbor sum loop, as the first line of the loop body (right after the opening `{` of the loop), add:

* `#pragma HLS UNROLL`

So the body of the `for (int i = 0; i < 9; i++)` loop starts with this pragma, followed by the `if (i == 4) continue;` and accumulation.

Keep:

* `#pragma HLS PIPELINE II=1` above the inner `x` loop.
* All `fifo_shiftreg_2` pragmas from Experiment 3.
* `#pragma HLS ARRAY_PARTITION variable=mat3 complete dim=1` after the `mat3` declaration.

**Synthesis:**

* Run synthesis.
* Record:

  * Achieved II for the inner `x` loop.
  * Latency.
  * Resource usage and max frequency.
* Compare with Step 4.1:

  * II should get closer to 1 if it wasn’t already.

---

## 7. Experiment 5 – BRAM-Based FIFO (fifo_bram)

### 7.1 Objective

Now switch to `fifo_bram` to see a *different* hardware implementation for the FIFO: circular buffer on RAM instead of a shift register.

### 7.2 Actions

1. In `gameoflife_compute`, change the three line buffers so they use `fifo_bram` instead of `fifo_shiftreg_2` (and not `fifo_shiftreg_1`).

   * Use `grid_width - 1` as the constructor parameter.
2. Inside the `fifo_bram` class:

   * Keep the logic as provided.
   * Optionally, you can guide the design toward BRAM explicitly using a pragma.

**Optional storage binding for clear BRAM usage**

**Location:**

* In `fifo_bram`, in the `private` section, immediately after the `data` array declaration (`T data[max_width];`).

**Action (optional but recommended for the lab):**

Add this line after the `data` array declaration in `fifo_bram`:

* `#pragma HLS BIND_STORAGE variable=data type=ram_t2p impl=bram`

This makes it explicit that `data` is mapped to BRAM.

Keep any loop or pipeline pragmas you already have in `gameoflife_compute` (for a fair comparison).

### 7.3 Synthesis and Measurement

* Run synthesis.
* Record latency, II, resource usage.
* Compare with:

  * `fifo_shiftreg_1` (Experiment 1).
  * Optimized `fifo_shiftreg_2` from Experiment 3 and 4.

You should see that `fifo_bram` uses far fewer LUTs/FFs and more BRAM, with good performance, but different access patterns and possibly different timing.

---

## 8. Run on Pynq

* Run package.
* Use vivado to build the system.
* Use the provided notebook to run the system.
* Notice the latency and compare with the reported latency from Vitis.

---

## 9. Final Comparison and Discussion

Build a summary table including at least:

* `fifo_shiftreg_1` (Experiment 1).
* `fifo_shiftreg_2` with no pragmas (Experiment 2).
* `fifo_shiftreg_2` with:

  * `BIND_STORAGE` only.
  * `BIND_STORAGE` + `INLINE`.
  * `BIND_STORAGE` + `INLINE` + `UNROLL`.
  * `BIND_STORAGE` + `INLINE` + `UNROLL` + `ARRAY_PARTITION`.
* Full pipelined system with optimized `mat3` (Experiment 4).
* `fifo_bram` version (Experiment 5).

For each, include:

* Latency (cycles).
* Achieved II of the inner `x` loop.
* LUTs, FFs, BRAMs.
* Estimated max frequency.

Then, discuss the following:

1. Why is `fifo_shiftreg_2` without pragmas so much slower than `fifo_shiftreg_1`, even though both conceptually “shift lines”?
2. Which pragma had the biggest impact on performance for `fifo_shiftreg_2`:

   * `#pragma HLS BIND_STORAGE variable=data type=ram_1p impl=lutram`
   * `#pragma HLS INLINE`
   * `#pragma HLS UNROLL`
   * `#pragma HLS ARRAY_PARTITION variable=data complete dim=1`
3. How does `#pragma HLS BIND_STORAGE variable=data type=ram_1p impl=lutram` differ in effect from
   `#pragma HLS ARRAY_PARTITION variable=data complete dim=1`?
4. Comparing optimized `fifo_shiftreg_2` and `fifo_bram`, which would you choose for:

   * Very large grids.
   * Very resource-constrained FPGAs.
     Explain with numbers from your table.
5. How did the pipelining and `mat3` optimizations (`PIPELINE`, `ARRAY_PARTITION` on `mat3`, `UNROLL` on the sum loop) influence the II and the total latency?

By the end, you’ll have watched three different FIFO designs give three different personalities to the same Game of Life accelerator, all steered by a handful of `#pragma HLS ...` lines.
